Declare AI Usage (Copilot) to understand some python syntax and libraries available, eg. tkinter, ttkbootstrap.

In [1]:
import numpy as np
import numpy.random as npr
import math
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
from ttkbootstrap.widgets import DateEntry

import ttkbootstrap as tb # for themes 

In [2]:
# geometric Brownian motion for stock prices
# S0 (float): initial index level
# r (float): constant risk-free rate
# sigma (float): constant volatility
# start_date (datetime): pricing date
# final_date (datetime): final date of pricing
# freq (str): frequency of pricing
# I (int): number of simulations

# gbm function that raises errors for some invalid inputs 
def gbm(S0, r, sigma, start_date, final_date, freq, I):
# raising errors for invalid inputs 
    if final_date <= start_date:
        raise ValueError("Final date must be after start date.")
    if S0 <= 0:
        raise ValueError("Initial price must be > 0.")
    if sigma < 0:
        raise ValueError("Volatility must be >= 0.")
    if I < 1:
        raise ValueError("Number of simulations must be >= 1.")

    time_grid = pd.date_range(start=start_date, end=final_date, freq=freq).to_pydatetime()
    if len(time_grid) < 2:
        raise ValueError("Time grid too short.")

    M = len(time_grid)
    S = np.zeros((M, I)) 
    S[0, :] = S0 
    for t in range(1, M):
        dt_frac = (time_grid[t] - time_grid[t - 1]).days / 365
        z = npr.standard_normal(I)
        S[t, :] = S[t - 1, :] * np.exp((r - 0.5 * sigma ** 2) * dt_frac + sigma * math.sqrt(dt_frac) * z)
    return (time_grid, S)

In [3]:
# function for reading inputs from the user and generate the stock price simulation
def getS():
    S0 = float(entry_s0.get().strip())
    r = float(entry_r.get().strip())
    sigma = float(entry_sigma.get().strip())
    I = int(entry_I.get().strip())

    # dates
    sd = date_start.get_date()
    ed = date_end.get_date()
    start_date = dt.datetime(sd.year, sd.month, sd.day)
    final_date = dt.datetime(ed.year, ed.month, ed.day)

    # mapping frequency and raise error 
    freq_choice = combo_freq.get().strip()
    freq_map = {"Daily": "D", "Weekly": "W", "Monthly": "ME"}
    freq = freq_map.get(freq_choice)
    if freq is None:
        raise ValueError("Frequency must be Daily / Weekly / Monthly.")
    return S0, r, sigma, start_date, final_date, freq, I

def get_paths():
    S0, r, sigma, start_date, final_date, freq, I = getS()
    return gbm(S0, r, sigma, start_date, final_date, freq, I)

# function for computing the result of simulation
def compute_result():
    try:
        time_grid, S = get_paths()
        mean_final = float(S[-1, :].mean())
    except Exception as exc:
        messagebox.showerror("Error", str(exc), parent=root)
        return

    messagebox.showinfo(
        "Result",
        f"The simulated price at the end of the given period is:\n{mean_final}",
        parent=root,
    )

# function for plotting graph 
def plot_graph():
    try:
        time_grid, S = get_paths()
    except Exception as exc:
        messagebox.showerror("Error", str(exc), parent=root)
        return

    ax.clear()
    ax.set_title("First 10 simulated paths")
    ax.set_xlabel("Time step")
    ax.set_ylabel("Price")

    n_show = min(10, S.shape[1])
    x = np.arange(S.shape[0])
    for j in range(n_show):
        ax.plot(x, S[:, j])

    ax.set_xlim(0, S.shape[0] - 1)
    canvas.draw()

# additional function for computing some statistics 
def compute_stats(final_prices):
    """Return mean/median/std/min/max of 1D array final_prices."""
    v = np.asarray(final_prices, dtype=float)
    if v.size == 0:
        raise ValueError("No values to compute statistics.")
    std = float(v.std(ddof=1)) if v.size > 1 else 0.0
    return {
        "mean": float(v.mean()),
        "median": float(np.median(v)),
        "std": std,
        "min": float(v.min()),
        "max": float(v.max()),
    }

def show_stats():
    try:
        _, S = get_paths()
        stats = compute_stats(S[-1, :])
    except Exception as exc:
        messagebox.showerror("Error", str(exc), parent=root)
        return

    msg = (
        "Final price statistics\n"
        f"Mean:   {stats['mean']:.4f}\n"
        f"Median: {stats['median']:.4f}\n"
        f"Std:    {stats['std']:.4f}\n"
        f"Min:    {stats['min']:.4f}\n"
        f"Max:    {stats['max']:.4f}\n"
    )
    messagebox.showinfo("Statistics", msg, parent=root)

# create the main window
root = tb.Window(themename="flatly")
main = ttk.Frame(root)
main.pack(fill="both", expand=True)
# make the window pop up at the front 
root.update_idletasks()
root.attributes("-topmost", True)
root.lift()
root.focus_force()
root.attributes("-topmost", False)
root.title("Stock price simulation")

# input label and entry
left = ttk.Frame(main)
left.pack(side="left", fill="y", padx=(0, 10))

right = ttk.Frame(main)
right.pack(side="right", fill="both", expand=True)

def add_row(row, label, widget):
    ttk.Label(left, text=label).grid(row=row, column=0, sticky="w", pady=6, padx=(0, 10))
    widget.grid(row=row, column=1, sticky="ew", pady=6)
    left.grid_columnconfigure(1, weight=1)

# entry box for parameters 
entry_s0 = ttk.Entry(left, width=18)
entry_r = ttk.Entry(left, width=18)
entry_sigma = ttk.Entry(left, width=18)
entry_I = ttk.Entry(left, width=18)

entry_s0.insert(0, "100")
entry_r.insert(0, "0.05")
entry_sigma.insert(0, "0.25")
entry_I.insert(0, "1000")

add_row(0, "Initial price (S0)", entry_s0)
add_row(1, "Short rate (r)", entry_r)
add_row(2, "Volatility (sigma)", entry_sigma)

# dates
date_start = DateEntry(left, width=16, dateformat="%Y-%m-%d")
date_end = DateEntry(left, width=16, dateformat="%Y-%m-%d")
today = dt.date.today()
date_start.set_date(today)
date_end.set_date(today + dt.timedelta(days=365))
add_row(3, "Pricing date", date_start)
add_row(4, "Final date", date_end)
    
combo_freq = ttk.Combobox(left, values=["Daily", "Weekly", "Monthly"], state="readonly", width=16)
combo_freq.set("Monthly")
add_row(5, "Frequency", combo_freq)

add_row(6, "Number of simulations (I)", entry_I)

# compute button
btns = ttk.Frame(left)
btns.grid(row=7, column=0, columnspan=2, sticky="w", pady=(12, 0))

tb.Button(btns, text="Compute", bootstyle="primary", command=compute_result).pack(side="left", padx=(0, 10))
tb.Button(btns, text="Plot graph", bootstyle="secondary", command=plot_graph).pack(side="left")
tb.Button(btns, text="Statistics", bootstyle="info", command=show_stats).pack(side="left")

# plot canvas
fig = Figure(figsize=(6.6, 4.8), dpi=100)
ax = fig.add_subplot(111)
ax.set_title("First 10 simulated paths")
ax.set_xlabel("Time step")
ax.set_ylabel("Price")

canvas = FigureCanvasTkAgg(fig, master=right)
canvas.get_tk_widget().pack(fill="both", expand=True)
canvas.draw()

# run the GUI loop
root.mainloop()